# 7. Event Metrics

`peyes.event_metrics` aggregates a whole sequence of `Event` objects into feature vectors, counts, rates, and
distributions — the kind of numbers you'd report for one recording, or compare across conditions/detectors.

In [1]:
import numpy as np
import peyes
import _helpers

d = _helpers.load_example_trial()
detector = peyes.create_detector("engbert", missing_value=np.nan, min_event_duration=4, pad_blinks_time=0)
labels, _ = detector.detect(
    t=d["t"], x=d["x"], y=d["y"], pixel_size_cm=d["pixel_size"], viewer_distance_cm=d["viewer_distance"],
)
events = peyes.create_events(
    labels=labels, t=d["t"], x=d["x"], y=d["y"], pupil=d["pupil"],
    pixel_size=d["pixel_size"], viewer_distance=d["viewer_distance"],
)

## Feature vectors

`durations`, `amplitudes`, `azimuths`, `center_pixels`, `start_times`, `end_times` each return one value per event,
across every label mixed together:

In [2]:
peyes.event_metrics.durations(events)[:10]

array([204.046,  60.01 , 222.044,  24.008,   4.003,   4.   ,   5.998,
        28.008, 420.082,  30.009])

`features_by_labels` groups those same features by label into one row per label — a quick per-label
overview:

In [3]:
peyes.event_metrics.features_by_labels(events)

,event_type,start_time,end_time,duration,distance,amplitude,azimuth,peak_velocity,median_velocity,min_velocity,cumulative_distance,cumulative_amplitude,center_pixel,pixel_std,dispersion,ellipse_area,is_outlier,outlier_reasons,count
label,,,,,,,,,,,,,,,,,,,
1,"[FIXATION, FIXATION, FIXATION, FIXATION, FIXAT...","[6.0, 274.055, 524.112, 536.112, 574.121, 1028...","[210.046, 496.099, 528.115, 542.11, 994.203, 1...","[204.046, 222.04399999999998, 4.00300000000004...","[7.3568447856400665, 40.1020579139026, 2.41105...","[0.2379620123242155, 1.297074064221906, 0.0779...","[99.9675565770277, 6.302340574375054, 194.3434...","[42.99341663285474, 43.67985083392221, 26.1453...","[13.650235829091333, 13.020897659332926, 19.51...","[1.0622503478065621, 0.902295664510711, 12.891...","[94.52818711168338, 103.50812839630332, 2.4473...","[3.0568558455388493, 3.3470915208993954, 0.079...","[(520.6592281553399, 418.31843883495134), (291...","[(1.3781321854123456, 2.85209201717164), (12.0...","[0.5222721072275128, 1.6010333727834978, 0.094...","[0.049608616072953325, 0.2652212731543597, 0.0...","[False, False, True, True, False, True, False,...","[[], [], [min_duration], [min_duration], [], [...",30
2,"[SACCADE, SACCADE, SACCADE, SACCADE, SACCADE, ...","[212.047, 498.103, 530.112, 544.112, 996.204, ...","[272.057, 522.111, 534.112, 572.12, 1026.213, ...","[60.01000000000002, 24.00799999999998, 4.0, 28...","[241.14681831817725, 127.84074746093285, 5.273...","[7.788051287274355, 4.133306045424129, 0.17057...","[178.51382994636012, 12.796403391978043, 197.5...","[158.49884524065953, 148.08059827011903, 45.31...","[79.26370813032068, 111.1090228195396, 40.7851...","[19.203812652612697, 21.94479260382554, 36.255...","[319.3043023942078, 136.62466465112453, 5.2767...","[10.300292822700372, 4.417032641819005, 0.1706...","[(347.81459677419355, 412.33902903225817), (38...","[(85.4484891508158, 3.520229042144439), (48.09...","[8.706069637324264, 5.139701022162855, 0.21398...","[2.715995618319964, 3.5106915338625404, 0.0065...","[False, False, True, False, False, True, False...","[[], [], [min_duration], [], [], [min_duration...",31
3,[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],0
4,[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],0
5,[BLINK],[2380.499],[2472.515],[92.01600000000008],[nan],[nan],[nan],[nan],[nan],[nan],[nan],[nan],"[(nan, nan)]","[(nan, nan)]",[nan],[nan],[False],[[]],1


## Counts, rates, and ratios

`counts` tallies events per label. `saccade_rate` / `blink_rate` give occurrences per second; `microsaccade_rate`
and `microsaccade_ratio` isolate small-amplitude saccades (below `max_amplitude` degrees, default 1.0):

In [4]:
print(peyes.event_metrics.counts(events))
print(f"saccade rate: {peyes.event_metrics.saccade_rate(events):.2f}/s")
print(f"blink rate: {peyes.event_metrics.blink_rate(events):.2f}/s")
print(f"microsaccade rate: {peyes.event_metrics.microsaccade_rate(events, max_amplitude=1.0):.2f}/s")
print(f"microsaccade ratio: {peyes.event_metrics.microsaccade_ratio(events, max_amplitude=1.0):.2%}")

label
1    30
2    31
3     0
4     0
5     1
Name: count, dtype: int64
saccade rate: 5.58/s
blink rate: 0.18/s
microsaccade rate: 2.34/s
microsaccade ratio: 41.94%


## Transition matrix

How often does one event label follow another? `transition_matrix` counts consecutive label pairs (or normalizes
each row to probabilities with `normalize_rows=True`):

In [5]:
peyes.event_metrics.transition_matrix(events, normalize_rows=True)

To,2,1,5
From,,,
1,1.0,0.000000,0.000000
2,0.0,0.966667,0.033333
5,1.0,0.000000,0.000000


## What's next

**[8 Event Matching & Match Evaluation](./8%20Event%20Matching%20%26%20Match%20Evaluation.ipynb)** — relating two
independent event sequences (e.g. a detector vs. a human rater) to each other, event by event.